##### HIGH LEVEL IMPLEMENTATION OF THE PAPER ' Improving Language Understanding by Generative Pre-Training (Radford et al., 2018 - GPT-1)' USING PYTORCH

here is the link to the paper: https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf

In [ ]:
import torch
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from collections import Counter, defaultdict
import torch.nn.functional as F
import re
import math
import time


In [ ]:
with open("input.txt", "r", encoding="utf-8") as f:
    unsupervised_texts = f.read().strip().split("\n\n")
    unsupervised_texts = unsupervised_texts[:3]

print("unsupervised_texts, input to pretrainer: \n", unsupervised_texts)

In [ ]:
df = pd.read_csv("imdb.csv")

label_map = {"negative": 0, "positive": 1}
df["label"] = df["sentiment"].map(label_map)

reviews = df["review"].tolist()
sentiments = df["label"].tolist()
# supervised_data = list(zip(reviews, sentiments))
reviews = reviews[:3]
sentiments = sentiments[:3]
print("imdb input to fine tuner:")
print("reviews:\n ", reviews)
print("labels:\n", sentiments)


In [ ]:
class VocabularyBuilder:
    def __init__(self):
        # Special tokens required for translation models
        self.word2idx = {
            "<PAD>": 0,
            "<UNK>": 1,
            "<SOS>": 2,
            "<EOS>": 3,
            "</w>": 4,
            "<EXTRACT>": 5,
        }
        self.idx2word = {v: k for k, v in self.word2idx.items()}
        self.merge_order = []
        self.num_words = len(self.word2idx)

        # This will hold our word frequency dictionary
        self.word_freqs = Counter()

    def preprocess_and_count(self, all_sentences):

        raw_word_counts = Counter()

        for sentence in all_sentences:
            sentence = sentence.lower()
            sentence = sentence.replace("\n", " ")
            sentence = re.sub(r"^[^:]+:\s*", "", sentence)
            sentence = re.sub(r'([.,!?;:"\'()])', r" \1 ", sentence)

            words = sentence.strip().split()
            raw_word_counts.update(words)

        # Convert words into spaces-separated characters with an end-of-word symbol
        self.word_freqs = Counter()
        for word, count in raw_word_counts.items():
            # Example: 'hug' -> 'h u g </w>'
            char_tuple = tuple(list(word) + ["</w>"])
            self.word_freqs[char_tuple] = count
        # Initialize basic single-character vocabulary
        self.build_initial_vocabulary()

    def build_initial_vocabulary(self):
        """Extracts unique base characters from our counted words."""
        for word_tuple in self.word_freqs.keys():
            for char in word_tuple:
                if char not in self.word2idx:
                    self.word2idx[char] = self.num_words
                    self.idx2word[self.num_words] = char
                    self.num_words += 1

    def get_pair_counts(self):

        pairs = defaultdict(int)
        for word_tuple, freq in self.word_freqs.items():
            for i in range(len(word_tuple) - 1):
                pair = (word_tuple[i], word_tuple[i + 1])
                pairs[pair] += freq
        # print("pairs: ", pairs)
        return pairs

    def merge_vocab(self, pair):

        new_word_freqs = Counter()
        bigram = re.escape(" ".join(pair))
        # Look for the pair with a boundary or space around it
        p = re.compile(r"(?<!\S)" + bigram + r"(?!\S)")
        # Save the merge order rule
        merged_string = "".join(pair)
        self.merge_order.append(pair)

        # Add the new merged token to the vocabulary if it isn't there
        if merged_string not in self.word2idx:
            self.word2idx[merged_string] = self.num_words
            self.idx2word[self.num_words] = merged_string
            self.num_words += 1

        # Update the character tuple configurations
        for word_tuple, freq in self.word_freqs.items():
            word_str = " ".join(word_tuple)
            new_word_str = p.sub(merged_string, word_str)
            new_word_tuple = tuple(new_word_str.split())
            new_word_freqs[new_word_tuple] = freq

        self.word_freqs = new_word_freqs

    def fit(self, all_sentences, num_merges):

        self.preprocess_and_count(all_sentences)

        for i in range(num_merges):
            pairs = self.get_pair_counts()
            if not pairs:
                break
            # Find the most common pair
            best_pair = max(pairs, key=pairs.get)
            self.merge_vocab(best_pair)
            # Optional progress information
            if (i + 1) % 500 == 0:
                print(f"Completed {i + 1} merges")

        print(f"Final vocabulary size: {self.num_words}")

        print(f"Total merges learned: {len(self.merge_order)}")

    def _word_tokenizer(self, word):
        word_tuple = tuple(list(word) + ["</w>"])

        for pair in self.merge_order:
            new_word = []
            i = 0
            while i < len(word_tuple):
                if (
                    i < len(word_tuple) - 1
                    and word_tuple[i] == pair[0]
                    and word_tuple[i + 1] == pair[1]
                ):
                    # when get match, Fuse them together
                    new_word.append(pair[0] + pair[1])
                    i += 2  # Skip the next character since we consumed it
                else:
                    new_word.append(word_tuple[i])
                    i += 1
            word_tuple = tuple(new_word)

        return word_tuple

    def tokenize(self, sentence, strip_speaker=False):
        sentence = sentence.lower()
        sentence = sentence.replace("\n", " ")
        sentence = re.sub(r'([.,!?;:"\'()])', r" \1 ", sentence)

        if strip_speaker:
            sentence = re.sub(r"^[^:]+:\s*", "", sentence)

        words = sentence.strip().split()

        tokens = []
        # 3. Process every word through the BPE tokenizer
        for word in words:
            bpe_tokens = self._word_tokenizer(word)
            tokens.extend(bpe_tokens)
        return tokens

    def numericalize(self, tokens):

        numerical_ids = [
            self.word2idx.get(token, self.word2idx["<UNK>"]) for token in tokens
        ]

        return numerical_ids


In [ ]:
unsupervised_tokenizer = VocabularyBuilder()
unsupervised_tokenizer.fit(unsupervised_texts, num_merges=5000)

In [ ]:
block_size = 128

all_sentences = []
for sentence in unsupervised_texts:
    tokenized_sentence = unsupervised_tokenizer.tokenize(sentence, strip_speaker=True)
    print("token: ", tokenized_sentence)
    all_sentences.extend(tokenized_sentence)


all_texts = [
    all_sentences[i : i + block_size] for i in range(0, len(all_sentences), block_size)
]

print(all_texts)

In [ ]:
class LanguageModelDataset(Dataset):
    def __init__(self, tokens, tokenizer):

        self.tokens = tokens
        self.tokenizer = tokenizer

    def __len__(self):

        return len(self.tokens)

    def __getitem__(self, idx):

        numeric_tokens = self.tokenizer.numericalize(self.tokens[idx])
        src_tensor = torch.tensor(
            numeric_tokens,
            dtype=torch.long,
        )

        return src_tensor

In [ ]:
class ClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):

        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):

        return len(self.texts)

    def __getitem__(self, idx):
        tokens = self.tokenizer.tokenize(self.texts[idx])

        numeric_tokens = self.tokenizer.numericalize(tokens)

        token_tensor = torch.tensor(
            [self.tokenizer.word2idx["<SOS>"]]
            + numeric_tokens
            + [self.tokenizer.word2idx["<EOS>"]]
            + [self.tokenizer.word2idx["<EXTRACT>"]],
            dtype=torch.long,
        )
        label_tensor = torch.tensor(self.labels[idx], dtype=torch.long)

        return (token_tensor, label_tensor)

In [ ]:
def collate_fn_for_unsupervised_learning(batch):

    padded_batch = pad_sequence(batch, padding_value=0, batch_first=True)

    return padded_batch


In [ ]:
def collate_fn_for_supervised_learning(batch):

    sequences, labels = zip(*batch)

    padded_seq = pad_sequence(sequences, padding_value=0, batch_first=True)

    return (padded_seq, torch.tensor(labels, dtype=torch.long))


In [ ]:
unsupervised_dataset = LanguageModelDataset(all_texts, unsupervised_tokenizer)


unsupervised_dataloader = DataLoader(
    unsupervised_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn_for_unsupervised_learning,
)

supervised_dataset = ClassificationDataset(reviews, sentiments, unsupervised_tokenizer)


supervised_dataloader = DataLoader(
    supervised_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn_for_supervised_learning,
)


src_batch = next(iter(unsupervised_dataloader))

print("\nSource batch shape:", src_batch.shape)

print("\nSource batch:")

print(src_batch)


src_batch, trg_batch = next(iter(supervised_dataloader))

print("\nSource batch shape:", src_batch.shape)

print("Target batch shape:", trg_batch.shape)

print("\nSource batch:")

print(src_batch)

print("\nTarget batch:")

print(trg_batch)

In [ ]:
class EmbeddingLayer(nn.Module):
    def __init__(self, vocab_size, d_model, max_seq_len=5000):
        super().__init__()
        self.vocab_size = vocab_size
        self.emb_dim = d_model
        self.max_seq_len = max_seq_len
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_embedding = nn.Embedding(max_seq_len, d_model)
        self.dropout = nn.Dropout(0.1)

    def forward(self, src):
        batch_size, seq_len = src.shape
        if seq_len > self.max_seq_len:
            raise ValueError(
                f"Sequence length ({seq_len}) exceeds max_seq_len ({self.max_seq_len})"
            )

        embedded = self.embedding(src)
        positions = torch.arange(0, seq_len, device=src.device).unsqueeze(0)
        position_emb = self.positional_embedding(positions)
        updated_embedded = embedded + position_emb

        output_embedding = self.dropout(updated_embedded)

        return output_embedding

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, h, d_model, d_k, d_v, max_seq_len=5000):
        super().__init__()
        self.h = h
        self.d_k = d_k
        self.d_v = d_v
        self.attention_dropout = nn.Dropout(0.1)
        self.Wq = nn.Linear(d_model, h * d_k)
        self.Wk = nn.Linear(d_model, h * d_k)
        self.Wv = nn.Linear(d_model, h * d_v)
        self.feedforward = nn.Linear(h * d_v, d_model)

        self.register_buffer(
            "causal_mask",
            torch.triu(torch.ones(max_seq_len, max_seq_len), diagonal=1).bool(),
        )

    def forward(
        self, query_source, key_source, value_source, mask=False, padd_mask=None
    ):
        batch_size = query_source.size(0)
        seq_len = query_source.size(1)

        # Project and reshape
        Q = (
            self.Wq(query_source)
            .view(batch_size, seq_len, self.h, self.d_k)
            .transpose(1, 2)
        )
        K = self.Wk(key_source).view(batch_size, -1, self.h, self.d_k).transpose(1, 2)
        V = self.Wv(value_source).view(batch_size, -1, self.h, self.d_v).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # Apply masks
        if mask:
            scores.masked_fill_(self.causal_mask[:seq_len, :seq_len], float("-inf"))

        if padd_mask is not None:
            scores.masked_fill_(padd_mask.unsqueeze(1).unsqueeze(2), float("-inf"))

        # Attention weights and output
        attention = F.softmax(scores, dim=-1)
        add_dropout = self.attention_dropout(attention)
        output = torch.matmul(add_dropout, V)

        # Combine heads and project
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        return self.feedforward(output)

In [ ]:
class AttentionSublayer(nn.Module):
    def __init__(self, h, d_model, d_k, d_v, dropout=0.1):
        super().__init__()
        self.multi_head_attention_sublayer = MultiHeadAttention(h, d_model, d_k, d_v)

        self.normalization = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self, src, query_source, key_source, value_source, mask=False, padd_mask=None
    ):
        attention_output = self.multi_head_attention_sublayer(
            query_source=query_source,
            key_source=key_source,
            value_source=value_source,
            mask=mask,
            padd_mask=padd_mask,
        )

        add_dropout = self.dropout(attention_output)
        add_residual = src + add_dropout
        output = self.normalization(add_residual)

        return output


In [ ]:
class FeedForwardSublayer(nn.Module):
    def __init__(self, d_model, d_ff=None, dropout=0.1):
        super().__init__()
        if d_ff is None:
            d_ff = 4 * d_model
        self.feedforward1 = nn.Linear(d_model, d_ff)
        self.feedforward2 = nn.Linear(d_ff, d_model)
        self.normalization = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.gelu = nn.GELU()

    def forward(self, src):
        forward1 = self.feedforward1(src)
        activated1 = self.gelu(forward1)
        forward2 = self.feedforward2(activated1)

        add_dropout = self.dropout(forward2)
        add_residual = src + add_dropout
        output = self.normalization(add_residual)

        return output


In [ ]:
class Decoder(nn.Module):
    def __init__(self, h, d_model, d_k, d_v, embedding, classification_dim, n_layers=6):
        super().__init__()
        self.embedding_layer = embedding
        self.classifier_dropout = nn.Dropout(0.1)
        self.classify = nn.Linear(d_model, classification_dim)
        self.layers = nn.ModuleList(
            [
                nn.ModuleDict(
                    {
                        "self_attention": AttentionSublayer(h, d_model, d_k, d_v),
                        "feed_forward": FeedForwardSublayer(d_model),
                    }
                )
                for layers in range(n_layers)
            ]
        )

    def forward(self, data, is_classify=False):
        trg_padd_mask = data == 0

        batch_size, sequence_len = data.shape

        decoded = self.embedding_layer(data)
        for layer in self.layers:
            self_attention = layer["self_attention"](
                src=decoded,
                query_source=decoded,
                key_source=decoded,
                value_source=decoded,
                mask=True,
                padd_mask=trg_padd_mask,
            )

            feedforward_output = layer["feed_forward"](self_attention)
            decoded = feedforward_output

        if is_classify:
            last_non_zero_sequence_lengths = (data != 0).sum(dim=1) - 1
            seq_range = torch.arange(batch_size, device=data.device)
            last_token_in_data = decoded[seq_range, last_non_zero_sequence_lengths]
            last_token = self.classifier_dropout(last_token_in_data)
            output = self.classify(last_token)
        else:
            output = torch.matmul(decoded, self.embedding_layer.embedding.weight.T)
        return output


In [ ]:
class GPT(nn.Module):
    def __init__(self, decoder, device):
        super().__init__()
        self.decoder = decoder
        self.device = device

    def forward(self, data, is_classify=False, n_layers=6):

        decoder_output = self.decoder(data, is_classify=is_classify, n_layers=n_layers)

        return decoder_output


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VOCAB_SIZE = unsupervised_tokenizer.num_words
CLASS_OF_SENTIMENTS = 2
D_MODEL = 768
FEED_FORWARD_INNER = 3072
D_V = 64
D_K = 64
H = 12
n_layers = 12
weight_normalization = (0, 0.02)


def initialize_weights(module):
    # Linear layers
    if isinstance(module, nn.Linear):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)

        if module.bias is not None:
            nn.init.zeros_(module.bias)

    # Token embeddings and positional embeddings
    elif isinstance(module, nn.Embedding):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)

    # LayerNorm
    elif isinstance(module, nn.LayerNorm):
        nn.init.ones_(module.weight)
        nn.init.zeros_(module.bias)


embedding_layer = EmbeddingLayer(
    vocab_size=VOCAB_SIZE, d_model=D_MODEL, max_seq_len=5000
)
dec = Decoder(
    h=H,
    d_model=D_MODEL,
    d_k=D_K,
    d_v=D_V,
    embedding=embedding_layer,
    classification_dim=CLASS_OF_SENTIMENTS,
    n_layers=n_layers,
)
model = GPT(decoder=dec, device=device).to(device)

model.apply(initialize_weights)


pretrain_lr_max = 2.5e-4
fine_tune_lr = 6.25e-5
beta_1 = 0.9
beta_2 = 0.98
epsilon = 1e-9
lambda_ = 0.5
label_smoothing = 0.1
warmup_steps = 4000
N_EPOCHS_PRETRAIN = 100
N_EPOCHS_FINETUNE = 10
total_pretrain_steps = N_EPOCHS_PRETRAIN * len(unsupervised_dataloader)
total_finetune_steps = N_EPOCHS_FINETUNE * len(supervised_dataloader)
pretrain_warmup_steps = int(0.05 * total_pretrain_steps)


optimizer_pretrain = torch.optim.AdamW(
    model.parameters(),
    lr=pretrain_lr_max,
    betas=(beta_1, beta_2),
    eps=epsilon,
    weight_decay=0.01,
)

# Fine-tuning Optimizer Setup (Lower LR as per paper)
optimizer_finetune = torch.optim.AdamW(
    model.parameters(),
    lr=fine_tune_lr,
    betas=(beta_1, beta_2),
    eps=epsilon,
    weight_decay=0.01,
)


def pretrain_lr_multiplier(
    step, warmup_steps=pretrain_warmup_steps, total_steps=total_pretrain_steps
):
    if step < warmup_steps:
        return step / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * progress))


def finetune_lr_multiplier(step, total_steps=total_finetune_steps, warmup_frac=0.002):
    warmup_steps = max(1, int(warmup_frac * total_steps))
    if step < warmup_steps:
        return step / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return max(0.0, 1.0 - progress)


pretrain_scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer_pretrain, lr_lambda=pretrain_lr_multiplier
)
fine_tune_scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer_finetune, lr_lambda=finetune_lr_multiplier
)

unsupervised_criterion = nn.CrossEntropyLoss(
    ignore_index=0,
    label_smoothing=label_smoothing,
)
supervised_criterion = nn.CrossEntropyLoss()

print(f"Model initialized successfully. Device: {device}")

In [ ]:
def unsupervised_training(model, dataloader, optimizer, criterion):

    model.train()
    epoch_loss = 0

    for data in dataloader:
        data = data.to(device)
        inputs = data[:, :-1]
        targets = data[:, 1:]
        optimizer.zero_grad()
        output = model(inputs, is_classify=False)

        flat_trg = targets.reshape(-1)
        flat_output = output.reshape(-1, VOCAB_SIZE)

        loss = criterion(flat_output, flat_trg)
        loss.backward()

        optimizer.step()
        pretrain_scheduler.step()
        epoch_loss += loss.item()

    return epoch_loss / len(dataloader)


for epoch in range(N_EPOCHS_PRETRAIN):
    start_time = time.time()

    train_loss = unsupervised_training(
        model, unsupervised_dataloader, optimizer_pretrain, unsupervised_criterion
    )
    end_time = time.time()

    print(f"Epoch: {epoch + 1:02} | Time: {end_time - start_time:.2f}s")
    print(
        f"\tTrain Loss: {train_loss:.3f} | Jarring Perplexity: {math.exp(train_loss):.3f}"
    )


In [ ]:
# def supervised_training(model, dataloader, optimizer, criterion):

#     model.train()
#     epoch_loss = 0

#     for text, label in dataloader:
#         src = text.to(device)
#         trg = label.to(device)

#         optimizer.zero_grad()
#         output = model(src, is_classify=True)

#         loss = criterion(output, trg)
#         loss.backward()

#         optimizer.step()
#         fine_tune_scheduler.step()
#         epoch_loss += loss.item()

#     return epoch_loss / len(dataloader)


# # supervised training

# for epoch in range(N_EPOCHS_FINETUNE):
#     start_time = time.time()

#     train_loss = supervised_training(
#         model, supervised_dataloader, optimizer_finetune, supervised_criterion
#     )
#     end_time = time.time()

#     print(f"Epoch: {epoch + 1:02} | Time: {end_time - start_time:.2f}s")
#     print(
#         f"\tTrain Loss: {train_loss:.3f} | Jarring Perplexity: {math.exp(train_loss):.3f}"
#     )

In [ ]:
def auxillary_training(
    model, dataloader, optimizer, unsupervised_criterion, supervised_criterion, lambda_
):

    model.train()
    epoch_loss = 0
    epoch_clf_loss = 0
    epoch_lm_loss = 0
    correct_predictions = 0
    total_predictions = 0
    for text, label in dataloader:
        l2_clf_src = text.to(device)
        l2_clf_trg = label.to(device)

        l1_inputs = l2_clf_src[:, :-1]
        l1_targets = l2_clf_src[:, 1:]

        optimizer.zero_grad()
        clf_output_1 = model(l1_inputs, is_classify=False)

        flat_trg = l1_targets.reshape(-1)
        flat_output = clf_output_1.reshape(-1, VOCAB_SIZE)
        L1 = unsupervised_criterion(flat_output, flat_trg)

        llm_output_2 = model(l2_clf_src, is_classify=True)

        L2 = supervised_criterion(llm_output_2, l2_clf_trg)

        L3 = L2 + lambda_ * L1
        L3.backward()

        optimizer.step()
        fine_tune_scheduler.step()

        # metrics
        predictions = torch.argmax(llm_output_2, dim=1)
        correct_predictions += (predictions == l2_clf_trg).sum().item()
        total_predictions += l2_clf_trg.size(0)

        epoch_loss += L3.item()
        epoch_clf_loss += L2.item()
        epoch_lm_loss += L1.item()

    # Calculate average metrics
    avg_loss = epoch_loss / len(dataloader)
    avg_clf_loss = epoch_clf_loss / len(dataloader)
    avg_lm_loss = epoch_lm_loss / len(dataloader)
    accuracy = correct_predictions / total_predictions

    return {
        "total_loss": avg_loss,
        "L2_classification_loss": avg_clf_loss,
        "L1_language_model_loss": avg_lm_loss,
        "accuracy": accuracy,
    }


for epoch in range(N_EPOCHS_FINETUNE):
    start_time = time.time()

    metrics = auxillary_training(
        model,
        supervised_dataloader,
        optimizer_finetune,
        unsupervised_criterion,
        supervised_criterion,
        lambda_,
    )
    end_time = time.time()

    print(f"Epoch: {epoch + 1:02} | Time: {end_time - start_time:.2f}s")
    print(f"\tTrain Loss: {metrics['total_loss']:.3f}")
    print(f"\tL2_classification_loss: {metrics['L2_classification_loss']:.3f} ")
    print(
        f"\tL1_language_model_loss: {metrics['L1_language_model_loss']:.3f} |  Jarring Perplexity: {math.exp(metrics['L1_language_model_loss']):.3f} "
    )
    print(f"\taccuracy: {metrics['accuracy']:.3f} ")

In [ ]:
def classify_sentence(sentence, device, tokenizer, model, max_len=500):
    model.eval()
    tokens = tokenizer.tokenize(sentence)
    numeric_tokens = tokenizer.numericalize(tokens)

    token_tensor = torch.tensor(
        [tokenizer.word2idx["<SOS>"]]
        + numeric_tokens
        + [tokenizer.word2idx["<EOS>"]]
        + [tokenizer.word2idx["<EXTRACT>"]],
        dtype=torch.long,
    )
    token_tensor = token_tensor.unsqueeze(0).to(device)
    with torch.no_grad():
        model_output = model(token_tensor, is_classify=True)
    prediction = model_output[-1, :].argmax().item()
    return prediction


test_sentences = [
    "This is a bad movie",
    "I really enjoyed this film",
    "The actors killed their roles",
    "I will love to sleep or be bored than to see this movie",
]
for sentence in test_sentences:
    prediction = classify_sentence(sentence, device, unsupervised_tokenizer, model)
    print("sentence: ", sentence)
    if prediction == 1:
        print("prediction: POSITIVE")
    else:
        print("prediction: NEGATIVE")